# VQPAINT tiny decoder (distilled VQGAN f16 decoder)

Small conv decoders from `vqgan_imagenet_f16_16384` tokens to RGB, distilled from the original decoder on on-the-fly token grids. Output: `/kaggle/working/tiny/`.

In [ ]:
S = dict(HOURS=1.2, BATCH=16, VARIANTS="A:64,64,64,32,16:2,2,2,1,1;B:64,64,48,24,12:2,2,2,1,1", LPIPS=1.0, LR=3e-4, GAN_W=0.0, GAN_START=0.4, CLIP_W=0.3, BRANCH="main")
# round 2 = resume from the previous version's output (this kernel is its own kernel source) + CLIP-faithfulness loss; HOURS are added
OUT = "/kaggle/working/tiny"
print(S)

In [ ]:
# setup: repo (research branch), taming-transformers, checkpoint (vqpaint-assets dataset or heibox), token grids (vqpaint-tokens dataset)
import glob, os, shutil, subprocess, sys, time
TMP, WORK = "/kaggle/tmp", "/kaggle/working"; os.makedirs(TMP, exist_ok=True)
REPO, DATA = f"{TMP}/VQPAINT", f"{TMP}/data"
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
def find_input(name):
    for root in ("/kaggle/input", "/kaggle/input/datasets"):
        for pat in (f"{root}/{name}*", f"{root}/*/{name}*", f"{root}/**/{name}*"):
            hits = sorted(glob.glob(pat, recursive=True))
            if hits: return hits[0]
    return None
sh("find /kaggle/input -maxdepth 4 | head -50 || true")
if not os.path.isdir(REPO): sh(f"git clone -q --branch {S['BRANCH']} --depth 1 https://github.com/NazarGol/tiny-vqgan.git {REPO}")
TAM = f"{REPO}/export/taming-transformers"
if not os.path.isdir(TAM): sh(f"git clone -q --depth 1 https://github.com/CompVis/taming-transformers.git {TAM}")
CK = f"{REPO}/export/checkpoints"; os.makedirs(CK, exist_ok=True)
ASSETS = find_input("vqpaint-assets"); print("assets dataset:", ASSETS)
for f in ("vqgan_imagenet_f16_16384.yaml", "vqgan_imagenet_f16_16384.ckpt"):
    dst = f"{CK}/{f}"
    if os.path.exists(dst): continue
    src = f"{ASSETS}/checkpoints/{f}" if ASSETS else None
    if src and os.path.exists(src): os.symlink(src, dst)
    else:
        url = "https://heibox.uni-heidelberg.de/f/274fb24ed38341bfa753/?dl=1" if f.endswith("yaml") else "https://heibox.uni-heidelberg.de/f/867b05fc8c4841768640/?dl=1"
        sh(f"curl -sSL '{url}' -o {dst}")
TOK = find_input("vqpaint-tokens"); print("tokens dataset:", TOK); assert TOK, "vqpaint-tokens dataset not mounted (see the listing above)"
if not os.path.isdir(DATA): shutil.copytree(TOK, DATA)
sh(f"ls -la {DATA} {CK}/ && nvidia-smi --query-gpu=name,memory.total --format=csv")
sh(f"{sys.executable} -m pip install -q lpips tokenizers scipy")
sh(f"{sys.executable} -m pip install -q onnxruntime-gpu==1.19.2 || {sys.executable} -m pip install -q onnxruntime")
for f in ("onnx/vision_model.onnx", "onnx/text_model.onnx", "tokenizer.json"):
    dst = f"{DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst): sh(f"curl -sSL https://huggingface.co/Xenova/mobileclip_s0/resolve/main/{f} -o {dst}")
import onnxruntime as ort; print("ORT providers:", ort.get_available_providers())
sh(f"{sys.executable} {REPO}/research/tiny/export_clip_vision.py --onnx {DATA}/vision_model.onnx --out {DATA}/clipx")   # differentiable CLIP for the faithfulness loss


In [ ]:
os.makedirs(OUT, exist_ok=True)
prev = sorted(glob.glob("/kaggle/input/**/vqpaint-tiny-decoder*/tiny/ckpt.pt", recursive=True))
if prev and not os.path.exists(f"{OUT}/ckpt.pt"): shutil.copy(prev[-1], f"{OUT}/ckpt.pt"); print("resuming from", prev[-1])

In [ ]:
cmd = f"{sys.executable} -u {REPO}/research/tiny/train_decoder.py --data {DATA} --out {OUT} --hours {S['HOURS']} --batch {S['BATCH']} --variants '{S['VARIANTS']}' --lpips {S['LPIPS']} --lr {S['LR']} --gan-w {S['GAN_W']} --gan-start {S['GAN_START']} --clip-w {S['CLIP_W']} --clip-ir {DATA}/clipx --clip {DATA}/vision_model.onnx"
print("$", cmd, flush=True)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
with open(f"{OUT}/train.log", "a") as log:
    for line in p.stdout:
        if "Warning" in line or "warn(" in line: continue
        print(line, end="", flush=True); log.write(line)
print("exit", p.wait())

In [ ]:
import json
print(json.dumps(json.load(open(f"{OUT}/eval.json")), indent=1)); sh(f"ls -la {OUT}")
if os.path.exists(f"{OUT}/eval_sheet.png"):
    from IPython.display import Image, display; display(Image(f"{OUT}/eval_sheet.png", width=700))